# Support Vector Classifiers

Here we try many support vector classifiers, with different kernels. 

In [151]:
import pandas as pd

learn_data = pd.read_csv("preprocess_train_v4.csv", header = None)
learn_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt',
       'LogSgot', 'Target']
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot,Target
0,48,0.511111,0.226640,4.615385,1.504077,5.641907,2.564949,4.304065,0
1,39,0.473684,-0.182383,3.115942,0.641854,5.192957,3.737670,4.127134,0
2,23,0.300000,-0.264120,3.100000,0.000000,5.356586,3.713572,4.382027,0
3,42,0.285714,-0.374875,3.018868,-0.356675,5.023881,3.555348,4.394449,0
4,54,0.504425,0.604032,4.250000,3.117950,6.324359,3.401197,3.610918,0


In [152]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.20, random_state = 42)

## Metrics

In [153]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

crossval_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])
validation_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear kernel (or no kernel)

We have seen with other linear classifiers that the performance is not very good because of two reasons:
- Excessive resampling: we might be resampling too much, and this may affect our predictive power by creating samples that do not correspond to the real data.

In [154]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC(class_weight = "balanced")
linear_model.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(linear_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	83	16
	0	108	153
Accuracy: 65.56%


In [155]:
confusion(np.array(y_val), pd.Series(linear_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	24	5
	0	23	38
Accuracy: 68.89%


In [156]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

linear_model = LinearSVC()
linsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", linear_model)])

n = 100
m = 10
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]
Cs = np.logspace(start = -1, stop = 2, num = n)

linsvc_search = GridSearchCV(estimator = linsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__class_weight' : weights},
                             scoring = 'f1_macro',
                             cv = 5)
linsvc_search.fit(X_train, y_train)
linsvc_search.best_params_

{'svc__C': 2.31012970008316, 'svc__class_weight': {0: 0.4, 1: 0.6}}

In [157]:
linsvc_search.best_score_

0.6465187319001321

In [158]:
from sklearn.model_selection import cross_validate

linsvc_C = linsvc_search.best_params_['svc__C']
linsvc_weights = linsvc_search.best_params_['svc__class_weight']
linsvc_best = LinearSVC(C = linsvc_C,
                        class_weight = linsvc_weights)
linsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", linsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(linsvc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Linear SVC", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.646519,0.657548,0.642417,0.702778


In [159]:
linsvc_pipeline.fit(X_train, y_train)
validation_df.loc["Linear SVC", :] = compute_metrics(y_val, linsvc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.602054,0.601187,0.603111,0.655556


## Gaussian kernel

In [160]:
from sklearn.svm import SVC

rbf_scale_model = SVC(kernel = "rbf", gamma = "scale", class_weight = "balanced")
rbf_scale_model.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_scale_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	56	43
	0	88	173
Accuracy: 63.61%


In [161]:
confusion(np.array(y_val), pd.Series(rbf_scale_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	14	15
	0	19	42
Accuracy: 62.22%


In [162]:
rbfsvc = SVC(kernel = "rbf")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc)])

n = 100
m = 10
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]
Cs = np.logspace(start = -1, stop = 2, num = n)
# gammas = np.logspace(start = -1, stop = 1, num = m) / X.shape[0]

rbfsvc_search = GridSearchCV(estimator = rbfsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__class_weight' : weights},
                                           #'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
rbfsvc_search.fit(X_train, y_train)
rbfsvc_search.best_params_

{'svc__C': 0.5722367659350217, 'svc__class_weight': {0: 0.3, 1: 0.7}}

In [163]:
rbfsvc_search.best_score_

0.6414793418815703

In [164]:
rbfsvc_C = rbfsvc_search.best_params_['svc__C']
rbfsvc_weights = rbfsvc_search.best_params_['svc__class_weight']
# rbfsvc_gamma = rbfsvc_search.best_params_['svc__gamma']
rbfsvc_best = SVC(kernel = "rbf",
                  C = rbfsvc_C,
                  gamma = "scale",
                  class_weight = rbfsvc_weights)
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(rbfsvc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Gaussian SVC", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.646519,0.657548,0.642417,0.702778
Gaussian SVC,0.641479,0.709385,0.668225,0.655556


In [165]:
rbfsvc_pipeline.fit(X_train, y_train)
validation_df.loc["Gaussian SVC", :] = compute_metrics(y_val, rbfsvc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian SVC,0.619214,0.676088,0.658848,0.622222
Linear SVC,0.602054,0.601187,0.603111,0.655556


## Polynomial kernel

In [166]:
poly_model = SVC(kernel = "poly", degree = 2, gamma = "scale", class_weight = "balanced")
poly_model.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	75	24
	0	124	137
Accuracy: 58.89%


In [167]:
confusion(np.array(y_val), pd.Series(poly_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	19	10
	0	27	34
Accuracy: 58.89%


In [171]:
polysvc = SVC(kernel = "poly", class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc)])

n = 100
m = 10
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]
Cs = np.logspace(start = -1, stop = 2, num = n)
# gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]
degrees = [2, 3]

polysvc_search = GridSearchCV(estimator = polysvc_pipeline,
                              param_grid = {'svc__C' : Cs,
                                            'svc__class_weight' : weights,
#                                             'svc__gamma' : gammas,
                                            'svc__degree' : degrees},
                              scoring = 'f1_macro',
                              cv = 5)
polysvc_search.fit(X_train, y_train)
polysvc_search.best_params_

# Results from before:
# {'svc__C': 0.35564803062231287,
#  'svc__degree': 3,
#  'svc__gamma': 0.2222222222222222}

{'svc__C': 4.977023564332111,
 'svc__class_weight': {0: 0.3, 1: 0.7},
 'svc__degree': 3}

In [172]:
polysvc_search.best_score_

0.6372899791988085

In [173]:
polysvc_C = polysvc_search.best_params_['svc__C']
# polysvc_gamma = polysvc_search.best_params_['svc__gamma']
polysvc_degree = polysvc_search.best_params_['svc__degree']
polysvc_weights = polysvc_search.best_params_['svc__class_weight']
polysvc_best = SVC(kernel = "poly",
                   C = polysvc_C,
                   gamma = "scale",
                   degree = polysvc_degree,
                   class_weight = polysvc_weights)
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc_best)])

cross_val_results = pd.DataFrame(cross_validate(polysvc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Polynomial SVC", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.646519,0.657548,0.642417,0.702778
Gaussian SVC,0.641479,0.709385,0.668225,0.655556
Polynomial SVC,0.63729,0.696596,0.657983,0.655556


In [174]:
polysvc_pipeline.fit(X_train, y_train)
validation_df.loc["Polynomial SVC", :] = compute_metrics(y_val, polysvc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
Gaussian SVC,0.619214,0.676088,0.658848,0.622222
Linear SVC,0.602054,0.601187,0.603111,0.655556


## Sigmoid

In [175]:
sig_model = SVC(kernel = "sigmoid", gamma = "scale", class_weight = "balanced")
sig_model.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(sig_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	59	40
	0	110	151
Accuracy: 58.33%


In [176]:
confusion(np.array(y_val), pd.Series(sig_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	13	16
	0	26	35
Accuracy: 53.33%


In [177]:
sigsvc = SVC(kernel = "sigmoid", class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc)])

n = 100
m = 10
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]
Cs = np.logspace(start = -1, stop = 2, num = n)
# gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]

sigsvc_search = GridSearchCV(estimator = sigsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__class_weight' : weights},
#                                            'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
sigsvc_search.fit(X_train, y_train)
sigsvc_search.best_params_

# Results from before:
# {'svc__C': 21.209508879201902, 'svc__gamma': 3.236329950002764e-05}

{'svc__C': 1.3219411484660293, 'svc__class_weight': {0: 0.4, 1: 0.6}}

In [178]:
sigsvc_search.best_score_

0.6562064915223207

In [181]:
sigsvc_C = sigsvc_search.best_params_['svc__C']
sigsvc_weights = sigsvc_search.best_params_['svc__class_weight']
# sigsvc_gamma = sigsvc_search.best_params_['svc__gamma']
sigsvc_best = SVC(kernel = "sigmoid",
                  C = sigsvc_C,
                  gamma = "scale",
                  class_weight = sigsvc_weights)
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(sigsvc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Sigmoid SVC", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
Linear SVC,0.646519,0.657548,0.642417,0.702778
Gaussian SVC,0.641479,0.709385,0.668225,0.655556
Polynomial SVC,0.63729,0.696596,0.657983,0.655556


In [182]:
sigsvc_pipeline.fit(X_train, y_train)
validation_df.loc["Sigmoid SVC", :] = compute_metrics(y_val, sigsvc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556
Gaussian SVC,0.619214,0.676088,0.658848,0.622222
Linear SVC,0.602054,0.601187,0.603111,0.655556


## Trying our best models on test dataset


In [183]:
test_data = pd.read_csv("preprocess_test_v4.csv", header = None)
test_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt', 'LogSgot']
test_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot
0,11,0.142857,-0.460075,3.000000,-0.356675,6.383507,3.258097,3.367296
1,62,0.500000,-0.172320,5.000000,0.587787,5.411646,4.234107,5.043425
2,60,0.285714,-0.460075,3.818182,-0.356675,5.159055,3.465736,2.639057
3,60,0.491228,0.226237,4.102564,1.740466,5.365976,6.021023,6.745236
4,48,0.222222,-0.260240,3.000000,-0.105361,5.164786,3.178054,3.988984


### Linear kernel

In [188]:
linsvc_pipeline.fit(X, y)

labels_lin = pd.DataFrame(columns = ['ID', 'Label'])
labels_lin['Label'] = pd.DataFrame(linsvc_pipeline.predict(test_data))
labels_lin['ID'] = labels_lin.index + 1
labels_lin.to_csv('new_predictions/linsvc_best.csv', index = False)

### Polynomial kernel

In [190]:
polysvc_pipeline.fit(X, y)

labels_poly = pd.DataFrame(columns = ['ID', 'Label'])
labels_poly['Label'] = pd.DataFrame(polysvc_pipeline.predict(test_data))
labels_poly['ID'] = labels_poly.index + 1
labels_poly.to_csv('new_predictions/polysvc_best.csv', index = False)

### Gaussian kernel

In [191]:
rbfsvc_pipeline.fit(X, y)

labels_rbf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rbf['Label'] = pd.DataFrame(rbfsvc_pipeline.predict(test_data))
labels_rbf['ID'] = labels_rbf.index + 1
labels_rbf.to_csv('new_predictions/rbfsvc_best.csv', index = False)

### Sigmoid kernel

In [192]:
sigsvc_pipeline.fit(X, y)

labels_sig = pd.DataFrame(columns = ['ID', 'Label'])
labels_sig['Label'] = pd.DataFrame(sigsvc_pipeline.predict(test_data))
labels_sig['ID'] = labels_sig.index + 1
labels_sig.to_csv('new_predictions/sigsvc_best.csv', index = False)

### Discrepancies between models

None at all between gaussian and sigmoid kernels, but a lot of discrepancy with the gaussian

In [194]:
(labels_sig == labels_poly).value_counts()

ID    Label
True  True     81
      False    35
Name: count, dtype: int64